# Homework 6: Coding

In the code spaces below, write code that answers each question. You can assume that the code spaces will all be executed in order. 

#### Exercise 1 

Implement the function `incrementer_gate` to return a custom gate which increments by an arbitrary positive integer. Look at the `increment_circuit` code below to see how `incrementer_gate` will be used. Do not modify `increment_circuit`. Check this week's demo code for how to create custom gates in Qiskit, and read the `increment_circuit` code for hints on how to do other useful things. 

I've started you off with two function stubs, `incrementer_gate` and `two_power_inc`. I recommend using `two_power_inc` to build incrementers for powers of two out of primitive gates and then to use those adders in combination to build incrementers for other numbers. Use only *X* gates and controlled *X* gates (multiple controls are fine). Do not use any built-in Qiskit gates for addition or other operations besides these. Build and use custom gates from these components however you wish, but any custom gates should be built only out of these primitives. 

In [3]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler
import math

def two_power_inc(power, qubit_count):
    circuit=QuantumCircuit(qubit_count)
    for i in range(qubit_count-1, power-1, -1):
        if i==power:
            circuit.x(i)
        else:
            circuit.mcx(list(range(power,i)),i)
    return circuit.to_gate()


def incrementer_gate(num, qubit_count):
    circuit=QuantumCircuit(qubit_count)
    for i in range(qubit_count):
        if (num>>i)&1:
            circuit.append(two_power_inc(i,qubit_count),list(range(qubit_count)))
    return circuit.to_gate()


In [7]:
# Do not modify this function. Implement incrementer_gate 
# above so that it works correctly with this function. 
def increment_circuit(start_val, inc_val, qubit_count=4):
    """
    Increment start_val by inc_val in a qubit_count sized circuit
    Return the sum as a decimal int
    """
    circuit = QuantumCircuit(qubit_count)

    bit_string = f"{start_val:08b}"
    for (idx, flip) in enumerate(bit_string[::-1]):
        if flip == '1':
            circuit.x(idx)

    inc_gate = incrementer_gate(inc_val, qubit_count)
    circuit.append(inc_gate, list(range(qubit_count)))
    circuit.measure_all()

    # Run the simulation
    backend=AerSimulator()
    sampler=Sampler(backend)
    transpiled_circuit = transpile(circuit, backend)
    job = sampler.run([transpiled_circuit], shots=1)
    result = job.result()
    bitstrings = result[0].data.meas.get_bitstrings()
    return int(bitstrings[0], 2)  

In [6]:
# Exercise 1 tests
# Leave this cell as is. Should run with no errors. 

assert increment_circuit(0, 3) == 3, f"Increment result incorrect: {increment_circuit(0, 3)}"
assert increment_circuit(3, 5) == 8, f"Increment result incorrect: {increment_circuit(3, 5)}"
assert increment_circuit(2, 6) == 8, f"Increment result incorrect: {increment_circuit(2, 6)}"
assert increment_circuit(15, 16, 6) == 31, f"Increment result incorrect: {increment_circuit(15, 16, 6)}"
assert increment_circuit(128, 110, 8) == 238, f"Increment result incorrect: {increment_circuit(128, 110, 8)}"

print("Increment circuit tests completed with no errors")

Increment circuit tests completed with no errors


/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_47511/2682902627.py:21: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler=Sampler(backend)
/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_47511/2682902627.py:21: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The

#### Exercise 2

For exercise 2, you will implement the `add_two_numbers` function that takes two positive integers, initializes two quantum registers to hold the numbers to be summed and a classical register to store the sum, calculates the sum and sends it to the classical register, and then uncomputes all qubit operations to leave all qubits in the 0 state. The classical register should be named `cr_sum` to work with the starter code.

Requirements:

- The code should be written in a flexible enough way that any positive integers can be correctly handled, although larger circuits will take longer to run. You do not need to test with numbers larger than the ones in the asserts provided.
- Your solution should re-use code from exercise 1 where possible.
- Your generated circuit should be big enough to handle the necessary summation comfortably, but no bigger. This will mean calculating the number of qubits/bits needed to hold your values (including the sum). The Python `math` library has a function `log2` that can help with this.
- As with exercise 1, use only *X* gates and controls. Do not use Qiskit's built in `.inverse()` method to invert your gates; if you need an inverse of a gate, build it from these primitives. 
- As you can see in the `add_two_numbers` function stub, the function will return a dictionary with `sum` and `final_state` values. The `sum` value will be the sum and the `final_state` value should always be 0. 

Below is a circuit diagram representing 6+8. Note that the `meas` register is automatically created when you use a `measure_all` operation in a circuit. So it is distinct from the registers listed above:

<img src="./img/circuit_3+4.png" width="800">

In [10]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

# You may define any helper functions you need

def add_two_numbers(num1, num2):
    # TODO: Build the addition circuit described in the instructions above
    # Read the starter code below carefully. Note that it requires a 
    # classical register called `cr_sum` for the sum and accessses a `meas`
    # register which means there needs to be a measure_all() added to the
    # circuit.
    
    # Your code begins here:
    n1=max(1,num1.bit_length())
    n2=max(1,(num1+num2).bit_length())
    a=QuantumRegister(n1,'a')
    b=QuantumRegister(n2,'b')
    cr_sum=ClassicalRegister(n2,'cr_sum')
    circuit=QuantumCircuit(a,b,cr_sum)
    for i in range(n1):
        if(num1>>i)&1:
            circuit.x(a[i])
    for i in range(n2):
        if(num2>>i)&1:
            circuit.x(b[i])
    for i in range(n1):
        for j in range(n2-1,i-1,-1):
            circuit.mcx([a[i]]+list(b[i:j]),b[j])
    circuit.measure(b,cr_sum)
    for i in range(n1-1,-1,-1):
        for j in range(i,n2):
            circuit.mcx([a[i]]+list(b[i:j]),b[j])
    for i in range(n1):
        if(num1>>i)&1:
            circuit.x(a[i])
    for i in range(n2):
        if(num2>>i)&1:
            circuit.x(b[i])
    circuit.measure_all()
    # Your code ends here
    ######################################

    backend = AerSimulator()
    sampler = Sampler(backend)
    transpiled_circuit = transpile(circuit, backend)
    job = sampler.run([transpiled_circuit], shots=1)
    result = job.result()

    sum_bs = result[0].data.cr_sum.get_bitstrings()
    sum = int(sum_bs[0], 2) 
    final_state_bs = result[0].data.meas.get_bitstrings()
    final_state = int(final_state_bs[0], 2) 
    
    return {'sum': sum, 'final_state': final_state}


In [9]:
# Exercise 2 tests
# Leave this cell as is. Should run with no errors. 

test1 = add_two_numbers(0, 3)
assert test1['sum'] == 3, f"Addition result incorrect: {test1['sum']}"
assert test1['final_state'] == 0, f"Final state not zeroed out: {test1['final_state']}"

test2 = add_two_numbers(5, 0)
assert test2['sum'] == 5, f"Addition result incorrect: {test2['sum']}"
assert test2['final_state'] == 0, f"Final state not zeroed out: {test2['final_state']}"

test3 = add_two_numbers(5, 1)
assert test3['sum'] == 6, f"Addition result incorrect: {test3['sum']}"
assert test3['final_state'] == 0, f"Final state not zeroed out: {test3['final_state']}"

test4 = add_two_numbers(7, 7)
assert test4['sum'] == 14, f"Addition result incorrect: {test4['sum']}"
assert test4['final_state'] == 0, f"Final state not zeroed out: {test4['final_state']}"

test5 = add_two_numbers(1, 513)
assert test5['sum'] == 514, f"Addition result incorrect: {test5['sum']}"
assert test5['final_state'] == 0, f"Final state not zeroed out: {test5['final_state']}"

test6 = add_two_numbers(256, 257)
assert test6['sum'] == 513, f"Addition result incorrect: {test6['sum']}"
assert test6['final_state'] == 0, f"Final state not zeroed out: {test6['final_state']}"

print("Adder circuit tests completed with no errors")

/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_47511/1112138942.py:48: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)
/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_47511/1112138942.py:48: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. T

Adder circuit tests completed with no errors


#### Exercise 3

For Exercise 3, you will implement `multiply_two_numbers`. You'll have 3 quantum registers: one for one input value, one for the second input value, and a third for the product. The size of the product register will need to be the sum of the sizes of the input registers in order to handle the size of the product of the numbers. Similarly to the previous exercise, you'll also create a classical register to receive the result of the operation. Also similarly to the previous exercise, you will uncompute all quantum operations so that the resulting quantum state (measured with `measure_all`) is 0. The output structure is analogous to the previous exercise (see the provided assert tests below).

Refer to the lecture slides for how multiplication can be carried out. The final product is made up of adding powers of two. The powers of two that are added correspond to the products of pairs of bits. Remember that multiplying powers of 2 (or any shared base) is equivalent to adding their exponents. So $2^n\cdot 2^m = 2^{n+m}$. Once again it will be convenient to use your `two_power_inc` function from xercise 1. 

The same requirements as Exercise 2 apply. 

In [ ]:

def multiply_two_numbers(num1, num2):
    # TODO:
    # Your code begins here:
    n1=max(1, num1.bit_length())
    n2=max(1, num2.bit_length())
    a=QuantumRegister(n1,'a')
    b=QuantumRegister(n2,'b')
    p=QuantumRegister(n1+n2,'p')
    c=ClassicalRegister(n1 + n2, 'cr_prod')
    circuit = QuantumCircuit(a, b, p, c)

    for i in range(n1):
        if num1&(1<<i):
            circuit.x(a[i])
    for i in range(n2):
        if num2&(1<< i):
            circuit.x(b[i])
    for i in range(n1):
        for j in range(n2):
            k=i+j
            for t in range(n1+n2-1,k-1,-1):
                controls=[a[i],b[j]]
                for x in range(k,t):
                    controls.append(p[x])
                circuit.mcx(controls,p[t])
    circuit.measure(p,c)
    for i in reversed(range(n1)):
        for j in reversed(range(n2)):
            k=i+j
            for t in range(k,n1+n2):
                controls=[a[i],b[j]]
                for x in range(k,t):
                    controls.append(p[x])
                circuit.mcx(controls,p[t])
    for i in range(n1):
        if num1&(1<<i):
            circuit.x(a[i])
    for i in range(n2):
        if num2&(1<<i):
            circuit.x(b[i])
            
    circuit.measure_all()

    backend=AerSimulator()
    sampler=Sampler(backend)
    job=sampler.run([transpile(circuit,backend)],shots=1)
    data=job.result()[0].data
    prod=int(data.cr_prod.get_bitstrings()[0],2)
    final_state=int(data.meas.get_bitstrings()[0],2)

    # Your code ends here.
    ######################################
    return {'prod': prod, 'final_state': final_state}



In [15]:
# Exercise 3 tests
# Leave this cell as is. Should run with no errors. 

test1 = multiply_two_numbers(0, 3)
assert test1['prod'] == 0, f"Addition result incorrect: {test1['prod']}"
assert test1['final_state'] == 0, f"Final state not zeroed out: {test1['final_state']}"

test2 = multiply_two_numbers(5, 0)
assert test2['prod'] == 0, f"Addition result incorrect: {test2['prod']}"
assert test2['final_state'] == 0, f"Final state not zeroed out: {test2['final_state']}"

test3 = multiply_two_numbers(1, 8)
assert test3['prod'] == 8, f"Addition result incorrect: {test3['prod']}"
assert test3['final_state'] == 0, f"Final state not zeroed out: {test3['final_state']}"

test4 = multiply_two_numbers(7, 1)
assert test4['prod'] == 7, f"Addition result incorrect: {test4['prod']}"
assert test4['final_state'] == 0, f"Final state not zeroed out: {test4['final_state']}"

test5 = multiply_two_numbers(7, 8)
assert test5['prod'] == 56, f"Addition result incorrect: {test5['prod']}"
assert test5['final_state'] == 0, f"Final state not zeroed out: {test5['final_state']}"

test6 = multiply_two_numbers(3, 29)
assert test6['prod'] == 87, f"Addition result incorrect: {test6['prod']}"
assert test6['final_state'] == 0, f"Final state not zeroed out: {test6['final_state']}"

print("Multiplication circuit tests completed with no errors")


Multiplication circuit tests completed with no errors


/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_47511/1181890079.py:48: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler=Sampler(backend)
/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_47511/1181890079.py:48: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The